# Cognito GCS Report Inventory

Extract one row per report under `gs://cognito-gcs/Cognito_new/reports_test_phase/`. The dataframe contains the molecule name, dimension name, report filename, full GCS path, and an `is_archived` flag for PDFs inside an `archived` folder.

In [11]:
import os
from pathlib import Path, PurePosixPath

import pandas as pd
from google.cloud import storage
from google.oauth2 import service_account

BUCKET_NAME = "cognito-gcs"
REPORT_PREFIX = "Cognito_new/reports/"
SERVICE_ACCOUNT_PATH = r"C:\Users\p00102977\Desktop\Projects\API\repo\cognito-dev-380506-6ca25463100d.json"


def get_gcs_client(service_account_path: str) -> storage.Client:
    """Create a GCS client from a service-account JSON file."""
    credentials_file = Path(service_account_path).expanduser()
    if not credentials_file.is_file():
        raise FileNotFoundError(
            f"Service-account JSON file was not found: {credentials_file}. "
            "Set SERVICE_ACCOUNT_PATH or GOOGLE_APPLICATION_CREDENTIALS."
        )

    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_file)
    )
    return storage.Client(project=credentials.project_id, credentials=credentials)


def extract_report_inventory(
    client: storage.Client,
    bucket_name: str = BUCKET_NAME,
    prefix: str = REPORT_PREFIX,
) -> pd.DataFrame:
    """List active and archived report metadata below the report prefix."""
    normalized_prefix = prefix.strip("/") + "/"
    rows = []

    for blob in client.list_blobs(bucket_name, prefix=normalized_prefix):
        relative_path = blob.name[len(normalized_prefix):]
        if not relative_path or relative_path.endswith("/"):
            continue

        path_parts = PurePosixPath(relative_path).parts
        if len(path_parts) < 2:
            continue

        report_name = path_parts[-1]
        is_archived = any(
            folder.lower() == "archived" for folder in path_parts[1:-1]
        )
        rows.append(
            {
                "molecule_name": path_parts[0],
                "dimension_name": PurePosixPath(report_name).stem,
                "report_name": report_name,
                "report_path": f"gs://{bucket_name}/{blob.name}",
                "is_archived": is_archived,
            }
        )

    columns = [
        "molecule_name",
        "dimension_name",
        "report_name",
        "report_path",
        "is_archived",
    ]
    return (
        pd.DataFrame(rows, columns=columns)
        .sort_values(
            ["molecule_name", "dimension_name", "is_archived"],
            ignore_index=True,
        )
    )

In [12]:
gcs_client = get_gcs_client(SERVICE_ACCOUNT_PATH)
report_inventory_df = extract_report_inventory(
    client=gcs_client,
    bucket_name=BUCKET_NAME,
    prefix=REPORT_PREFIX,
)

report_inventory_df.head()

,molecule_name,dimension_name,report_name,report_path,is_archived
0,Acmopatide,20260712-144642_1.Primary Market Entry Horizon,20260712-144642_1.Primary Market Entry Horizon...,gs://cognito-gcs/Cognito_new/reports/Acmopatid...,False
1,Acmopatide,20260712-144815_Primary Market Entry Horizon,20260712-144815_Primary Market Entry Horizon.pdf,gs://cognito-gcs/Cognito_new/reports/Acmopatid...,False
2,Acmopatide,20260712-144943_Patent Strength and Invalidity...,20260712-144943_Patent Strength and Invalidity...,gs://cognito-gcs/Cognito_new/reports/Acmopatid...,False
3,Acmopatide,20260712-145016_Patent Thicket and Circumventi...,20260712-145016_Patent Thicket and Circumventi...,gs://cognito-gcs/Cognito_new/reports/Acmopatid...,False
4,Acmopatide,20260712-145146_Global Launch Sequencing & Arb...,20260712-145146_Global Launch Sequencing & Arb...,gs://cognito-gcs/Cognito_new/reports/Acmopatid...,False


In [13]:
report_inventory_df.to_csv(r'report_check_main.csv', index = False)